# Phát hiện bất thường trong dữ liệu số

Notebook khởi đầu của ML Academy VN. **Các kết quả bên dưới là kết quả thật khi chạy bằng Python 3 + scikit-learn lúc phát triển** (số có thể lệch nhẹ giữa các phiên bản thư viện). Hãy chạy lại, chỉnh sửa và hoàn thành các bước:

1. Sinh dữ liệu chuẩn rồi chèn 2% điểm bất thường có nhãn.
2. Áp dụng z-score, IQR và IsolationForest.
3. So sánh Precision, Recall, F1 của từng phương pháp.
4. Giải thích vì sao accuracy không phù hợp ở đây.

Baseline: Dự đoán toàn bộ là bình thường · Chỉ số: Precision, Recall, F1

Đây là **minh họa phát hiện trên chính dữ liệu fit** với outlier tổng hợp rất dễ tách, không phải đánh giá tổng quát hóa trên log production. Threshold/contamination đã được định trước; muốn triển khai cần baseline train và tập đánh giá thời gian độc lập, không ước lượng thống kê trên test.


In [1]:
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score
rng = np.random.default_rng(0)
normal = rng.normal(0, 1, 980); anom = rng.choice([-1, 1], 20) * rng.uniform(4, 6, 20)
x = np.concatenate([normal, anom]); y = np.r_[np.zeros(980), np.ones(20)]

In [2]:
z = np.abs((x - x.mean()) / x.std()) > 3
q1, q3 = np.percentile(x, [25, 75]); iqr = q3 - q1
ii = (x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)
if_ = IsolationForest(contamination=0.02, random_state=0).fit_predict(x.reshape(-1, 1)) == -1
for name, p in {'z-score>3': z, 'IQR': ii, 'IsolationForest': if_}.items():
    print(f'{name:16s} precision={precision_score(y, p):.3f} recall={recall_score(y, p):.3f} F1={f1_score(y, p):.3f}')

z-score>3        precision=0.909 recall=1.000 F1=0.952
IQR              precision=0.714 recall=1.000 F1=0.833
IsolationForest  precision=1.000 recall=1.000 F1=1.000


## Việc của bạn

- Thử thêm một mô hình hoặc siêu tham số khác và so sánh công bằng bằng cross-validation.
- Viết 3-5 câu nhận xét: chênh lệch nhỏ giữa các mô hình có đáng tin không?